# Analyzing GitHub Issues with the REST API

This notebook walks through how to use the GitHub REST API to fetch, transform, and analyze issues from any public repository. We'll use `serpapi/public-roadmap` as our example.

## Setup

Install dependencies:
```bash
pip install requests pandas altair
```

You'll need a GitHub personal access token. Create a **fine-grained token** at https://github.com/settings/tokens with read access to public repos, then set it as an environment variable:
```bash
export GITHUB_TOKEN=ghp_yourtoken
```

In [1]:
import os
import re
from datetime import datetime, timezone

import requests
import altair as alt
import pandas as pd

BASE_URL = "https://api.github.com"
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")

# Change these to analyze a different repo
OWNER = "serpapi"
REPO = "public-roadmap"

## 1. Fetching Issues

The GitHub REST API endpoint `GET /repos/{owner}/{repo}/issues` returns issues **and** pull requests mixed together. We need to:
- Filter out PRs by checking for the `"pull_request"` key
- Paginate with `per_page=100` (the maximum allowed)
- Handle GitHub's ~1,000 result pagination limit by fetching open and closed issues separately

In [3]:
def fetch_issues_by_state(owner, repo, state, headers):
    """Fetch all issues for a given state (open/closed), handling pagination."""
    issues = []
    page = 1

    while True:
        response = requests.get(
            f"{BASE_URL}/repos/{owner}/{repo}/issues",
            headers=headers,
            params={"state": state, "per_page": 100, "page": page},
        )

        # GitHub returns 422 when pagination exceeds ~1000 results
        if response.status_code == 422:
            break

        # Rate limit exceeded
        if response.status_code in (403, 429):
            raise Exception(
                "GitHub API rate limit exceeded. "
                "Wait a few minutes and try again, or check your GITHUB_TOKEN."
            )

        response.raise_for_status()
        data = response.json()

        if not data:
            break

        for issue in data:
            if "pull_request" not in issue:
                issues.append(issue)

        page += 1

    return issues


def fetch_all_issues(owner, repo):
    """Fetch all issues from the repo, excluding pull requests."""
    headers = {}
    if GITHUB_TOKEN:
        headers["Authorization"] = f"Bearer {GITHUB_TOKEN}"

    # Fetch open and closed separately to maximize results.
    # GitHub caps pagination at ~1000 results per query,
    # so splitting by state lets us get up to 2000 issues.
    open_issues = fetch_issues_by_state(owner, repo, "open", headers)
    closed_issues = fetch_issues_by_state(owner, repo, "closed", headers)

    return open_issues + closed_issues

In [4]:
raw_issues = fetch_all_issues(OWNER, REPO)
print(f"Fetched {len(raw_issues)} issues from {OWNER}/{REPO}")

Fetched 1910 issues from serpapi/public-roadmap


Let's look at what a raw issue looks like from the API:

In [5]:
# Inspect the first issue to understand the structure
sample = raw_issues[0]
print(f"Issue #{sample['number']}: {sample['title']}")
print(f"State: {sample['state']}")
print(f"Created: {sample['created_at']}")
print(f"Labels: {[l['name'] for l in sample.get('labels', [])]}")

Issue #3653: [Yelp Place API] Searches returning fully empty for valid results
State: open
Created: 2026-03-26T12:06:12Z
Labels: ['status: prioritized', 'type: bug']


## 2. Transforming the Data

We convert each raw issue into a structured record with:
- **service**: Extracted from the title prefix using regex (e.g., `[Google Search API]` → `"Google Search API"`)
- **status/type**: Parsed from label prefixes like `"status: queued"` or `"type: bug"`
- **age_days**: Days since the issue was created

In [6]:
def transform_issue(issue):
    """Convert a raw GitHub issue dict into a structured record."""
    created = datetime.fromisoformat(issue["created_at"].replace("Z", "+00:00"))
    age_days = (datetime.now(timezone.utc) - created).days
    labels = [label["name"] for label in issue.get("labels", [])]

    # Extract service from title prefix like "[Google Search]"
    match = re.search(r"\[(.+?)\]", issue["title"])
    service = match.group(1) if match else "General"

    # Extract status and type from label prefixes
    status = next((l.split(": ", 1)[1] for l in labels if l.startswith("status:")), "none")
    type_ = next((l.split(": ", 1)[1] for l in labels if l.startswith("type:")), "none")

    return {
        "number": issue["number"],
        "title": issue["title"],
        "state": issue["state"],
        "created_at": created.strftime("%Y-%m-%d"),
        "age_days": age_days,
        "labels": labels,
        "service": service,
        "status": status,
        "type": type_,
    }

In [7]:
records = [transform_issue(issue) for issue in raw_issues]
df = pd.DataFrame(records)
df.head(10)

,number,title,state,created_at,age_days,labels,service,status,type
0,3653,[Yelp Place API] Searches returning fully empt...,open,2026-03-26,0,"[status: prioritized, type: bug]",Yelp Place API,prioritized,bug
1,3652,[Google Travel Explore API] Valid airline code...,open,2026-03-26,0,"[status: queued, type: bug]",Google Travel Explore API,queued,bug
2,3650,[Google Search API] Missing top_stories sectio...,open,2026-03-25,0,"[status: queued, type: bug]",Google Search API,queued,bug
3,3649,[Google Search API] Missing certain content fr...,open,2026-03-25,0,"[status: freezer, type: improvement]",Google Search API,freezer,improvement
4,3645,[Google Search API] Missing results when `gl` ...,open,2026-03-24,2,"[status: prioritized, type: bug]",Google Search API,prioritized,bug
5,3644,[Google Search API] Allow switching between li...,open,2026-03-24,2,"[status: freezer, type: feature]",Google Search API,freezer,feature
6,3643,[Yandex Images API] Fully empty results,open,2026-03-24,2,"[status: queued, type: bug]",Yandex Images API,queued,bug
7,3642,[Google Jobs API] Job results missing descript...,open,2026-03-23,2,"[status: wip, type: bug]",Google Jobs API,wip,bug
8,3641,[New API] API to programmatically upgrade/down...,open,2026-03-23,2,"[status: freezer, type: feature]",New API,freezer,feature
9,3640,[Google Search API] Not parsing certain `time`...,open,2026-03-23,2,"[status: queued, type: bug]",Google Search API,queued,bug


## 3. Analysis

### Overview Metrics

In [8]:
total = len(df)
open_count = len(df[df["state"] == "open"])
closed_count = len(df[df["state"] == "closed"])

print(f"Total Issues: {total}")
print(f"Open: {open_count}")
print(f"Closed: {closed_count}")

Total Issues: 1910
Open: 913
Closed: 997


### Issues by Status

In [9]:
status_df = df["status"].value_counts().reset_index()
status_df.columns = ["status", "count"]

alt.Chart(status_df).mark_bar(cornerRadiusTopLeft=4, cornerRadiusTopRight=4).encode(
    x=alt.X("status:N", sort="-y", title=""),
    y=alt.Y("count:Q", title="Issues"),
    color=alt.value("#4A90D9"),
    tooltip=["status:N", "count:Q"],
).properties(width=500, height=350, title="Issues by Status")

alt.Chart(...)

### Issues by Type

In [10]:
type_df = df[df["type"] != "none"]["type"].value_counts().reset_index()
type_df.columns = ["type", "count"]

alt.Chart(type_df).mark_arc(innerRadius=60, outerRadius=120).encode(
    theta=alt.Theta("count:Q"),
    color=alt.Color("type:N", legend=alt.Legend(title="Type")),
    tooltip=["type:N", "count:Q"],
).properties(width=400, height=350, title="Issues by Type")

alt.Chart(...)

### Issues Over Time

In [11]:
df["month"] = pd.to_datetime(df["created_at"]).dt.to_period("M").astype(str)
monthly_state = df.groupby(["month", "state"]).size().reset_index(name="count")

alt.Chart(monthly_state).mark_area(opacity=0.6).encode(
    x=alt.X("month:N", title="", axis=alt.Axis(labelAngle=-45)),
    y=alt.Y("count:Q", title="Issues"),
    color=alt.Color("state:N", legend=alt.Legend(title="State")),
    tooltip=["month:N", "state:N", "count:Q"],
).properties(width=700, height=350, title="Issues Opened Over Time")

alt.Chart(...)

### Top 15 Services

In [12]:
service_df = df["service"].value_counts().head(15).reset_index()
service_df.columns = ["service", "count"]

alt.Chart(service_df).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4).encode(
    x=alt.X("count:Q", title="Issues"),
    y=alt.Y("service:N", sort="-x", title=""),
    color=alt.value("#4A90D9"),
    tooltip=["service:N", "count:Q"],
).properties(width=500, height=450, title="Top 15 Services")

alt.Chart(...)

### Open Issue Aging

In [13]:
def age_bucket(days):
    """Categorize issue age into human-readable buckets."""
    if days < 7:
        return "< 7d"
    elif days < 30:
        return "7-30d"
    elif days < 90:
        return "30-90d"
    elif days < 180:
        return "90-180d"
    elif days < 365:
        return "180-365d"
    else:
        return "365d+"


open_df = df[df["state"] == "open"].copy()
open_df["age_bucket"] = open_df["age_days"].apply(age_bucket)

bucket_order = ["< 7d", "7-30d", "30-90d", "90-180d", "180-365d", "365d+"]
aging_df = open_df["age_bucket"].value_counts().reindex(bucket_order, fill_value=0).reset_index()
aging_df.columns = ["bucket", "count"]

alt.Chart(aging_df).mark_bar(cornerRadiusTopLeft=4, cornerRadiusTopRight=4).encode(
    x=alt.X("bucket:N", sort=bucket_order, title=""),
    y=alt.Y("count:Q", title="Issues"),
    color=alt.value("#4A90D9"),
    tooltip=["bucket:N", "count:Q"],
).properties(width=500, height=400, title="Open Issue Aging Distribution")

alt.Chart(...)